# Import Packages

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
import json
import re
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

In [ ]:
from tensorflow.keras import layers, models, optimizers, Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout,BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical  # for 1-hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.utils import shuffle
from tqdm import tqdm
from sklearn.linear_model import SGDClassifier



from transformers import pipeline, BertTokenizer, TFBertModel



# sentences = ["I am not having a great day"]

# model_outputs = classifier(sentences)
# print(model_outputs[0])

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# load scam dataset
scam1_df = pd.read_csv("hf://datasets/BothBosu/Scammer-Conversation/gen_conver_noIdentifier_1000.csv")

splits = {'train': 'scam-dialogue_train.csv', 'test': 'scam-dialogue_test.csv'}
scam2_df = pd.read_csv("hf://datasets/BothBosu/scam-dialogue/" + splits["train"])
scam2_df.drop(columns=['type'])
scam2_df.rename(columns={"dialogue":'conversation'}, inplace=True)
scam2_df.head()

#1. Data Preprocessing

We write a function for preprocessing the data

`line_by_line`: It decomposes the conversation into setences and removes the speakers' names from the text,  meanwhile, keeping track of the which conversation it is and the speakers in a dataframe.

## 1.1 Scam datasets


In [ ]:
def line_by_line(df, conversation_column, preserve_columns, speaker_a, speaker_b):
    """
    Process a DataFrame to split conversation lines into separate rows and add additional columns.

    Parameters:
    df (pd.DataFrame): The original DataFrame.
    conversation_column (str): The name of the column containing the conversation text.
    preserve_columns (list): List of columns to preserve in the new DataFrame.
    speaker_a (str): The string identifying speaker A.
    speaker_b (str): The string identifying speaker B.

    Returns:
    pd.DataFrame: A new DataFrame with each conversation line as a separate row and additional columns.
    """

    # Function to split and combine conversation lines
    def split_and_combine_conversation(conversation):
        # Adjust the regex pattern to use the provided speaker identifiers
        parts = re.split(r'({}|{})'.format(re.escape(speaker_a), re.escape(speaker_b)), conversation)
        combined_utterances = []
        current_speaker = None
        current_utterance = ""

        for part in parts:
            if part.startswith(speaker_a):
                if current_speaker == 'A':
                    current_utterance += " "
                else:
                    if current_utterance:
                        combined_utterances.append((current_speaker, current_utterance.strip()))
                    current_speaker = 'A'
                    current_utterance = ""
                current_utterance += part[len(speaker_a):].strip()
            elif part.startswith(speaker_b):
                if current_speaker == 'B':
                    current_utterance += " "
                else:
                    if current_utterance:
                        combined_utterances.append((current_speaker, current_utterance.strip()))
                    current_speaker = 'B'
                    current_utterance = ""
                current_utterance += part[len(speaker_b):].strip()
            else:
                current_utterance += " " + part.strip()

        if current_utterance:
            combined_utterances.append((current_speaker, current_utterance.strip()))

        return combined_utterances

    # Create a list to store the new rows
    new_rows = []

    for index, row in df.iterrows():
        combined_utterances = split_and_combine_conversation(row[conversation_column])
        utterance_id = 0

        for speaker, utterance in combined_utterances:
            if speaker:
                new_row = {col: row[col] for col in preserve_columns}
                new_row[conversation_column] = utterance
                new_row['conversation_id'] = index
                new_row['utterance_id'] = utterance_id
                new_row['speaker'] = speaker
                new_rows.append(new_row)
                utterance_id += 1

    # Create a new DataFrame from the new rows
    new_df = pd.DataFrame(new_rows)
    new_df[conversation_column] = new_df[conversation_column].str.lower()
    return new_df

In [ ]:
scam1_line_by_line = line_by_line(scam1_df, 'conversation', ['label'], 'Person A:', 'Person B:')
scam2_line_by_line = line_by_line(scam2_df, 'conversation', ['label'], 'caller:', 'receiver:')
scam2_line_by_line['conversation_id'] = scam2_line_by_line['conversation_id'] + (scam1_line_by_line['conversation_id'].iloc[-1] + 1)
scam2_line_by_line

Final datasets for training:
* `extract_speaker_df1`: The whole conversation as a string, scam label, and speaker sequece vector. To be used in training MLP in model 1.
* `line_by_line`: Conversation seperated into lines. Will be labeled with emotion by pre-trained BERT (model 2).

In [ ]:
line_by_line_df = pd.concat([scam1_line_by_line, scam2_line_by_line]).reset_index(drop=True)

## 1.2 Extract emotions

Get the emotion label using a pretrained bert model
store in the google drive

The following codes use pretrained bert to get the emotion labels. (It takes some time to run everytime, so we directly store it in drive)

In [ ]:
# pretrained roberta-model
emotion_classifier = pipeline(task="text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None, device=0)
sentences = line_by_line_df['conversation'].tolist()
# get the emotions labels for sentences
results = emotion_classifier(sentences)

# store them in df
labels = [sublist[0]['label'] for sublist in results]
line_by_line_df['emotions'] = labels
line_by_line_df = line_by_line_df.dropna(subset=['conversation'])

# store in google drive
csv_path = '/content/drive/MyDrive/COMP4211/project/line_by_line.csv'
line_by_line_df.to_csv(csv_path, index=False)


In [ ]:
setence = ["The machine learning project is fun."]
results = emotion_classifier(setence)
labels =[ [item['label'] for item in sublist] for sublist in results][0]
probs = [[item['score'] for item in sublist] for sublist in results][0]
df = pd.DataFrame([probs],index = ['probability'],columns = labels)

In [ ]:

# emotion_classifier = pipeline(task="text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None, device=0)
# def visualize_prediction(num_samples):
#   sentences = line_by_line_df['conversation'].tolist()[0:num_samples]
#   results = emotion_classifier(sentences)
#   converted_results = [ [' '.join(f"{value:.3f}" if isinstance(value, float) else str(value) for value in item.values()) for item in sublist] for sublist in results]
#   display(pd.DataFrame(converted_results))
# visualize_prediction(3)

In [ ]:
# Create the one-hot encoding dictionary
emotions = [
    "admiration",
    "amusement",
    "anger",
    "annoyance",
    "approval",
    "caring",
    "confusion",
    "curiosity",
    "desire",
    "disappointment",
    "disapproval",
    "disgust",
    "embarrassment",
    "excitement",
    "fear",
    "gratitude",
    "grief",
    "joy",
    "love",
    "nervousness",
    "optimism",
    "pride",
    "realization",
    "relief",
    "remorse",
    "sadness",
    "surprise",
    "neutral"
]

# Create one-hot encoding dictionary
one_hot_dict = {emotion: [1 if i == j else 0 for i in range(len(emotions))] for j, emotion in enumerate(emotions)}

# Define the function to one-hot encode a list of emotions
def one_hot_encode_emotions(emotion_list):
    return [one_hot_dict[emotion] for emotion in emotion_list]


In [ ]:
# the line_by_line_df can be retrived from the google drive
csv_path = '/content/drive/MyDrive/COMP4211/project/line_by_line.csv'
line_by_line_df = pd.read_csv(csv_path)
line_by_line_df['conversation'] = line_by_line_df['conversation'].astype(str)

In [ ]:
# group the line by line together by their id
group_line_by_line_df = line_by_line_df.groupby('conversation_id', sort=False).agg({
    'conversation':lambda x:' '.join(x),
    'speaker': list,
    'emotions':list,
    'label': lambda x: x.iloc[0]
})
group_line_by_line_df.reset_index(inplace=True)
group_line_by_line_df['one_hot_encoded'] = group_line_by_line_df['emotions'].apply(one_hot_encode_emotions)

In [ ]:
# add the number of lines in each conversation
group_line_by_line_df['length'] = group_line_by_line_df['speaker'].apply(len)
group_line_by_line_df

In [ ]:
print(group_line_by_line_df['length'].mode())

group_line_by_line is our final dataset

## 1.3 Spliting dataset for training

We will split our dataset `group_line_by_line_df` into 3 parts.
One for pretraining the first 3 models, the second one for the last regression model, and the final one is for testing.

In [ ]:
# split dataset into three partitions
def shuffle_split(df, partition_sizes):
    """
    Shuffle and split a DataFrame into partitions of specified sizes.

    Args:
        df (pd.DataFrame): Input DataFrame to be shuffled and split.
        partition_sizes (list): List of fractions for partition sizes. Should sum to 1.0.

    Returns:
        list: List of DataFrame partitions.
    """
    if not np.isclose(sum(partition_sizes), 1.0):
        raise ValueError("Partition sizes must sum to 1.0")

    # Shuffle the DataFrame
    shuffled_df = df.sample(frac=1, random_state=4211).reset_index(drop=True)

    # Calculate the number of rows for each partition, ensuring all data is included
    total_rows = len(shuffled_df)
    partition_counts = [int(size * total_rows) for size in partition_sizes]

    # Adjust the last partition to include any leftover rows
    partition_counts[-1] += total_rows - sum(partition_counts)

    # Calculate partition indices
    partition_indices = np.cumsum([0] + partition_counts)

    # Split the shuffled DataFrame into partitions
    df_parts = [shuffled_df.iloc[partition_indices[i]:partition_indices[i + 1]].reset_index(drop=True) for i in range(len(partition_sizes))]

    return df_parts

partition_sizes = [0.5, 0.2, 0.3]
data1, data2, data3 = shuffle_split(group_line_by_line_df, partition_sizes)

In [ ]:
data1.head()

Prepare dataset for Longformer+MLP

In [ ]:
mlp_data = data1[['conversation', 'label']]

Prepare dataset for LSTM

`select_elements`: It will combine the emotion sequence from the same speaker.

In [ ]:
# prepare dataset for LSTMs
def select_elements(row, letter):
  return [val2 for val1, val2 in zip(row['speaker'], row['one_hot_encoded']) if val1 == letter]

# sentences spoken by speaker A
A_df = data1.apply(lambda row: select_elements(row, 'A'), axis=1).to_frame(name='emotion_sequence')
A_df['label'] = data1['label']

# sentences spoken by speaker B
B_df = data1.apply(lambda row: select_elements(row, 'B'), axis=1).to_frame(name='emotion_sequence')
B_df['label'] = data1['label']
B_df


In [ ]:
# first 10 data
display(A_df.iloc[0:10])
display(B_df.iloc[0:10])

# 2. Training

## 2.1 Longformer+MLP Training

We use Longformer to get a good embedding of conversation text and feed it in a MLP for classification task.

In [ ]:
# Load pre-trained Longformer model and preprocessor from TensorFlow Hub
from transformers import LongformerTokenizer, TFLongformerModel
# Initialize the tokenizer and model
tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096')
long_model = TFLongformerModel.from_pretrained('allenai/longformer-base-4096')



In [ ]:
# Function to get pooler_output for a batch of texts
# Input: a list of sentences (s,)
# output: tensor (s,768)
def get_pooler_outputs(texts):
    # Ensure texts is a list of sentences
    if isinstance(texts, str):
        texts = [texts]

    # Tokenize inputs
    encoded_inputs = tokenizer(
        texts,
        return_tensors='tf',
        max_length=4096,
        truncation=True,
        padding=True
    )

    # Get model outputs
    outputs = long_model(**encoded_inputs)
    pooler_outputs = outputs.pooler_output  # Shape: (batch_size, 768)

    return pooler_outputs


In [ ]:
# def check_empty_entries(texts):
#   empty_indices = [index for index, text in enumerate(texts) if text.strip() == ""]
#   return empty_indices
# check_empty_entries(mlp_data)
# mlp_train_X, mlp_val_X, mlp_train_y, mlp_val_y = train_test_split(mlp_data['conversation'].tolist(), mlp_data['label'].tolist(), test_size=0.2, random_state=4211)
# check_empty_entries(mlp_train_X)
# check_empty_entries(mlp_val_X)
# len(mlp_val_X)

We convert conversations to embeddings batch-wise since the computation resource is limited.

In [ ]:
# split the data
mlp_train_X, mlp_val_X, mlp_train_y, mlp_val_y = train_test_split(mlp_data['conversation'].tolist(), mlp_data['label'].tolist(), test_size=0.2, random_state=4211)

# convert the conversation to embeddings batch-wise
def convert_embeddings_batchwise(texts, batch_size=32):
  num_batches = len(texts)//batch_size + (1 if len(texts)% batch_size != 0 else 0)
  all_embeddings = []

  for i in range(num_batches):

    batch_texts = texts[i*batch_size:(i+1)*batch_size]
    batch_embeddings = get_pooler_outputs(batch_texts).numpy()
    all_embeddings.extend(batch_embeddings)

  return np.array(all_embeddings)




In [ ]:
mlp_train_embeddings = convert_embeddings_batchwise(mlp_train_X, 32)
mlp_val_embeddings = convert_embeddings_batchwise(mlp_val_X, 32)


# Function to save embeddings to a file
def save_embeddings(embeddings, file_path):
    with open(file_path, 'wb') as file:
        pickle.dump(embeddings, file)

save_embeddings(mlp_train_embeddings, '/content/drive/My Drive/COMP4211/project/mlp_train_embeddings.pkl')
save_embeddings(mlp_val_embeddings, '/content/drive/My Drive/COMP4211/project/mlp_val_embeddings.pkl')

In [ ]:
# def compare_embeddings(embed1, embed2):
#   return np.allclose(embed1, embed2)

We've already computed and stored them in google drive, so we can directly retrive them

In [ ]:
# Function to load embeddings from a file
def load_embeddings(file_path):
    with open(file_path, 'rb') as file:
        embeddings = pickle.load(file)
    return embeddings

mlp_train_embeddings = load_embeddings('/content/drive/My Drive/COMP4211/project/mlp_train_embeddings.pkl')
mlp_val_embeddings = load_embeddings('/content/drive/My Drive/COMP4211/project/mlp_val_embeddings.pkl')
print(mlp_train_embeddings.shape, mlp_val_embeddings.shape)

In [ ]:
# Model definition (Model 1 MLP)
model1 = Sequential([
    Dense(128, activation='relu', input_shape=(768,)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Training parameters
epochs = 15
batch_size = 16

# Convert labels to numpy array
mlp_train_y = np.array(mlp_train_y)
mlp_val_y = np.array(mlp_val_y)

# Define the ModelCheckpoint callback to save the best model based on validation accuracy
checkpoint_filepath = '/content/drive/My Drive/COMP4211/project/best_mlp_model.weights.h5'
model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

# Train the model with the callback
model1.fit(
    mlp_train_embeddings, mlp_train_y,
    validation_data=(mlp_val_embeddings, mlp_val_y),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[model_checkpoint_callback]
)

# Load the best weights after training
model1.load_weights(checkpoint_filepath)

# Now, the model has the best weights based on validation accuracy


In [ ]:
model1.summary()

In [ ]:
# a function for displaying confusion matrix
def display_confusion_matrix(model, val_data, val_labels):
  val_pred = model.predict(val_data)
  val_pred = np.array(val_pred)>=0.5
  cm = confusion_matrix(val_labels, val_pred)

  accuracy = accuracy_score(val_labels, val_pred)
  f1 = f1_score(val_labels, val_pred, average='weighted')
  plt.figure(figsize=(10,7))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
  plt.xlabel("Prediction")
  plt.ylabel("True Labels")
  plt.title('Confusion Matrix')
  plt.show()

  print(f"Accuracy: {accuracy:.4f}")
  print(f"F1 score: {f1}")




In [ ]:
display_confusion_matrix(model1, mlp_val_embeddings, mlp_val_y)

## 2.2 LSTMs Training

We train 2 LSTM models (scammer/victim) for scam classification with one-hot encoded emotions sequences as inputs.

In [ ]:
A_train, A_test, B_train, B_test = train_test_split(A_df, B_df, test_size=0.2, random_state=4211)

In [ ]:
print(type(A_train))
B_test.head()

In [ ]:
print(A_train.shape, A_test.shape)

In [ ]:
def generate_batches_and_pad(data, labels, batch_size):
    """
    Generate mini-batches and pad sequences.

    Args:
        data (list of lists): List of sequences.
        labels (list): List of labels corresponding to the sequences.
        batch_size (int): Size of each mini-batch.

    Returns:
        padded_data: Padded data for each batch.
        padded_labels: Labels for the batches.
    """
    padded_data = pad_sequences(data, padding='post', dtype='float32')
    return padded_data, np.array(labels)

def build_rnn_model():
    """
    Build and compile an RNN (LSTM) model.

    Returns:
        model: Compiled RNN model.
    """
    model = Sequential([
        LSTM(6, input_shape=(None, 28), return_sequences=False),
        BatchNormalization(),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def train_rnn(model_type, training_data, training_labels, val_data, val_labels, epochs=10, batch_size=32):
    """
    Train an RNN (LSTM) model on sequences with varying lengths.

    Args:
        model_type (str): The type of RNN model ('A' or 'B').
        training_data (list of lists): List of sequences, where each sequence is a list of vectors.
        training_labels (list): Binary labels for each sequence.
        val_data (list of lists): List of validation sequences.
        val_labels (list): Binary labels for each validation sequence.
        epochs (int): Number of training epochs.
        batch_size (int): Size of each mini-batch.

    Returns:
        model: Trained RNN model.
    """

    # Generate and pad batches for training and validation data
    padded_training_data, padded_training_labels = generate_batches_and_pad(training_data, training_labels, batch_size)
    padded_val_data, padded_val_labels = generate_batches_and_pad(val_data, val_labels, len(val_labels))

    # Build the RNN model
    model = build_rnn_model()

    # Adjust the checkpoint file path based on the model type
    checkpoint_filepath = f'/content/drive/My Drive/COMP4211/project/best_rnn_model_{model_type}.weights.h5'
    model_checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    )

    # Train the model using Keras's fit method
    model.fit(
        padded_training_data, padded_training_labels,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(padded_val_data, padded_val_labels),
        callbacks=[model_checkpoint_callback]
    )

    # Load the best weights after training
    model.load_weights(checkpoint_filepath)

    return model



In [ ]:
# Train the model
model3 = train_rnn('A', A_train['emotion_sequence'], A_train['label'], A_test["emotion_sequence"], A_test['label'], epochs=10, batch_size=16)


In [ ]:
model3.summary()

In [ ]:
model4 = train_rnn('B',B_train['emotion_sequence'], B_train['label'],B_test['emotion_sequence'],B_test['label'],  epochs=15, batch_size=16)


In [ ]:
model4.summary()

It's crucial  to pad data before sending into RNN (LSTM).

In [ ]:
# notice that we have to padd the data before we send it to model!
def pad_data(data):
  return pad_sequences(data, padding='post', dtype = 'float32')

In [ ]:
display_confusion_matrix(model3, pad_data(A_test['emotion_sequence']), np.array(A_test['label']))
display_confusion_matrix(model4, pad_data(B_test['emotion_sequence']), np.array(B_test['label']))

## 2.3 Logistic Regression Training

In [ ]:
def log_reg_df(data_embeddings, A_df, B_df):
  prob1 = model1.predict(data_embeddings).reshape(-1)
  prob3 = model3.predict(A_df).reshape(-1)
  prob4 = model4.predict(B_df).reshape(-1)
  X = pd.DataFrame()
  X['prob1'] = prob1
  X['prob3'] = prob3
  X['prob4'] = prob4
  return X

In [ ]:
# data trainsformed by model1
data2_embeddings = convert_embeddings_batchwise(data2['conversation'].tolist())
prob1 = model1.predict(data2_embeddings).reshape(-1)

In [ ]:
# data trainsformed by model3 and model4
A_df2 = data2.apply(lambda row: select_elements(row, 'A'), axis=1).to_frame(name='emotion_sequence')
B_df2 = data2.apply(lambda row: select_elements(row, 'B'), axis=1).to_frame(name='emotion_sequence')

A_df2 = pad_data(A_df2['emotion_sequence'])
B_df2 = pad_data(B_df2['emotion_sequence'])

prob3 = model3.predict(A_df2).reshape(-1)
prob4 = model4.predict(B_df2).reshape(-1)

In [ ]:
X = pd.DataFrame()
X['prob1'] = prob1
X['prob3'] = prob3
X['prob4'] = prob4
y = data2['label']
display(X.head())

In [ ]:
train_X, val_X, train_y, val_y = train_test_split(X,y, test_size = 0.2, random_state=4211)

In [ ]:
log_reg = SGDClassifier(loss='log_loss', penalty='l2', max_iter=1000, tol=1e-3)
log_reg.fit(train_X, train_y)
print("Coefficients (weights)", log_reg.coef_)
print("Intercept (bias)", log_reg.intercept_)

In [ ]:


# Your specified directory
directory = '/content/drive/My Drive/COMP4211/project/'

# Ensure the directory exists
os.makedirs(directory, exist_ok=True)

# File path
file_path = os.path.join(directory, 'log_reg_weights.json')

# Your weights
data = {
    "coefficients": log_reg.coef_.tolist(),
    "intercept": log_reg.intercept_.tolist()
}

# Save weights to JSON file
with open(file_path, 'w') as json_file:
    json.dump(data, json_file)


In [ ]:
display_confusion_matrix(log_reg, val_X, val_y)

# 3. Evaluating Performance

We use data3 for testing the performance of different (combinations of) models

In [ ]:
data3_embeddings = convert_embeddings_batchwise(data3['conversation'].tolist())
A_df3 = data3.apply(lambda row: select_elements(row, 'A'), axis=1).to_frame(name='emotion_sequence')
B_df3 = data3.apply(lambda row: select_elements(row, 'B'), axis=1).to_frame(name='emotion_sequence')
A_df3 = pad_data(A_df3['emotion_sequence'])
B_df3 = pad_data(B_df3['emotion_sequence'])

## 3.1 Model 1 only

In [ ]:
display_confusion_matrix(model1, data3_embeddings, data3['label'])

## 3.2 Model 3 only

In [ ]:
display_confusion_matrix(model3, A_df3, data3['label'])

## 3.3 Model 4 only


In [ ]:
display_confusion_matrix(model4, B_df3, data3['label'])

## 3.4 Model 5

In [ ]:
X = log_reg_df(data3_embeddings, A_df3, B_df3)
display_confusion_matrix(log_reg, X, data3['label'])


# 4. Discussion about Model 5 performance

## 4.1 Loading weights

We load the weights stored previously.

In [ ]:
model1 = Sequential([
    Dense(128, activation='relu', input_shape=(768,)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model1.load_weights("/content/drive/My Drive/COMP4211/project/best_mlp_model.weights.h5")
model3 = Sequential([
        LSTM(6, input_shape=(None, 28), return_sequences=False),
        BatchNormalization(),
        Dense(1, activation='sigmoid')
    ])
model3.load_weights('/content/drive/My Drive/COMP4211/project/best_rnn_model_A.weights.h5')
model4 = Sequential([
        LSTM(6, input_shape=(None, 28), return_sequences=False),
        BatchNormalization(),
        Dense(1, activation='sigmoid')
    ])
model4.load_weights('/content/drive/My Drive/COMP4211/project/best_rnn_model_B.weights.h5')


In [ ]:
# Your specified directory
directory = '/content/drive/My Drive/COMP4211/project/'

# File path
file_path = os.path.join(directory, 'log_reg_weights.json')

# Load the weights from the JSON file
with open(file_path, 'r') as json_file:
    weights = json.load(json_file)

model5 = SGDClassifier(loss='log_loss', penalty='l2', max_iter=1000, tol=1e-3, fit_intercept=True, random_state=4211)
model5.fit(X, data3['label']) # fake fitting
model5.coef_ = np.array(weights['coefficients'])
model5.intercept_ = np.array(weights['intercept'])

## 4.2 Collect predictions

We store the predictions of the four models in `Summary`

In [ ]:
X = log_reg_df(data3_embeddings, A_df3,B_df3)
Summary = X.copy(deep=True)
Summary['pred of M1'] = (model1.predict(data3_embeddings)>=0.5).astype(int)
Summary['pred of M3'] = (model3.predict(A_df3)>=0.5).astype(int)
Summary['pred of M4'] = (model4.predict(B_df3)>=0.5).astype(int)
Summary['pred of M5'] = model5.predict(X)
Summary['label']= data3['label']
Summary

## 4.3 Saved by Model 5

We see how many wrong prediction from Model 1 are **saved** by Model 5.



In [ ]:
wrong_pred1 = Summary[Summary['pred of M1']!=Summary['label']]
display(wrong_pred1)
print(f"There are in total {wrong_pred1.shape[0]} misclassifications from model 1.")

In [ ]:
fixed_by_M5 = wrong_pred1[wrong_pred1['pred of M5']==wrong_pred1['label']]
display(fixed_by_M5)
print(f"There are {fixed_by_M5.shape[0]} wrong classifications of M1 fixed by M5.")

In fact there are over 2/3 predictions are fixed after considering the outputs of emotions models.

## 4.4 Extra mistakes made by Model 5

Next, We see how many wrong predictions made by Model 5. Furthermore, how many of them Model 1 will also gives wrong predictions.


In [ ]:
wrong_pred5 = Summary[Summary['pred of M5'] != Summary['label']]
display(wrong_pred5)
print(f"There are in total {wrong_pred5.shape[0]} misclassifications from model 5.")
m1_also_wrong = wrong_pred5[wrong_pred5['pred of M1']!=wrong_pred5['label']]
display(m1_also_wrong)
print(f"There are in total {m1_also_wrong.shape[0]} misclassifications which is also wrong from model 1.")

We can see that among the misclassifications of Model 5, there is about 2/3 them Model 1 will also make the wrong predictions.